In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

from dotenv import load_dotenv
from collections.abc import Sequence
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

import flax.jax_utils as flax_utils
import flax.linen as nn
import grain.python as grain
import jax
import numpy as np
from absl import logging
from connectomics.jax import checkpoint, training
from etils import epath
from orbax import checkpoint as ocp

import zapbench.models.util as model_util
from zapbench.ts_forecasting import heads, input_pipeline, train
from zapbench.ts_forecasting.configs import infer, mean, linear, timemix, tsmixer, tide

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])

load_dotenv()
PATH = os.getenv("ROOT_PATH")
LOG_PATH = os.getenv("LOG_PATH")

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  # Set all spines invisible
  for spine in ax.spines.values():
    spine.set_visible(False)
  # Hide all ticks and tick labels
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
config = mean.get_config("dataset_name=subject_17,timesteps_input=4")
print(config.train_specs[0]['timeseries']['transform']['output'][0]['index_array']) #2045 5024

In [ ]:
2466-36, 2430-2045

In [ ]:
from zapbench.ts_forecasting import data_source

train_source = data_source.ConcatenatedTensorStoreTimeSeries(*[
    input_pipeline._build_merged_data_source(
        series=series,
        timesteps_input=config.timesteps_input,
        timesteps_output=config.timesteps_output,
        prefetch=config.prefetch,
        sequential=config.sequential_data_source,
    )
    for series in config.train_specs
])

In [ ]:
train_source[0]['timeseries_input'][0]

In [ ]:
import tensorstore as ts

x = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_17_traces.zarr'
    # gs://zapbench-release/volumes/20240930/traces/'
}).result().read().result()

In [ ]:
x.shape

In [ ]:
print(np.all(x[2431]==train_source[386]['timeseries_input'][0]))
print(np.all(x[2466]==train_source[386]['timeseries_output'][-1]))

In [ ]:
print(np.all(x[4855]==train_source[387]['timeseries_input'][0]))

In [ ]:
transform = config.train_specs[0]['timeseries']['transform']

x = [t[0] for t in transform['output'][0]['index_array']]

In [ ]:
x

In [ ]:
config = mean.get_config("dataset_name=240930_traces,timesteps_input=4")
print(config.train_specs[3])

In [ ]:
from zapbench import data_utils
# print(data_utils.get_condition_bounds(1, dataset_name="subject_17"))
print(data_utils.get_condition_intervals(4, dataset_name="subject_17"))
splits = ['train', 'val', 'test', 'test_holdout']
for split in splits:
  print(split)
  # print(data_utils.adjust_condition_bounds_for_split(split, 650, 2421, 4))
  window_size = data_utils.calculate_window_size(4)
  valid_timesteps = data_utils.build_valid_timesteps(((2470, 2748), (5280, 5552)), 32)
  print(data_utils.adjust_valid_timesteps_for_split(valid_timesteps, split, 4)[0], data_utils.adjust_valid_timesteps_for_split(valid_timesteps, split, 4)[-1])

In [ ]:
print(data_utils.adjust_valid_timesteps_for_split(valid_timesteps, 'test_holdout', 4))

In [ ]:
import tensorstore as ts

x = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/traces/',
}).result().read().result()

In [ ]:
from zapbench.ts_forecasting import data_source

train_source = data_source.ConcatenatedTensorStoreTimeSeries(*[
    input_pipeline._build_merged_data_source(
        series=series,
        timesteps_input=config.timesteps_input,
        timesteps_output=config.timesteps_output,
        prefetch=config.prefetch,
        sequential=config.sequential_data_source,
    )
    for series in config.train_specs
])

In [ ]:
print(np.all(x[2431]==train_source[386]['timeseries_input'][0]))
print(np.all(x[2466]==train_source[386]['timeseries_output'][-1]))

In [ ]:
i = 418
print(np.all(x[i+1]==train_source[i]['timeseries_input'][0]))
i = 419
print(np.all(x[i+1]==train_source[i]['timeseries_input'][0]))

In [ ]:
print(np.all(x[650]==train_source[419]['timeseries_input'][0]))

Also check inference

In [ ]:
import jax
from zapbench.ts_forecasting import heads, input_pipeline, train
from zapbench.ts_forecasting.configs import infer, mean
from connectomics.jax import training

infer_config = infer.get_config()
# config = mean.get_config('dataset_name=240930_traces,timesteps_input=4')
config = mean.get_config('dataset_name=subject_17,timesteps_input=4')
config.update(infer_config)

head = heads.create_head(config)
rng = training.get_rng(config.seed)
rng, infer_rng = jax.random.split(rng)
infer_source = input_pipeline.create_inference_source_with_transforms(config)
infer_key = jax.random.fold_in(key=infer_rng, data=1)

In [ ]:
for infer_idx_set in config.infer_idx_sets:
  name, idx_list = (infer_idx_set[k] for k in ('name', 'idx_list'))
  idx_list = config.infer_idx_sets[-1]['idx_list']
  infer_metrics = None
  print(name)
  # train_state = train.merge_batch_stats(train_state)
  # for i, idx in enumerate(idx_list):
  #   print(i)
    # infer_source[idx]

In [ ]:
print(config.infer_idx_sets[-1]['idx_list'])

In [ ]:
import tensorstore as ts

x = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_17_traces.zarr'
    # gs://zapbench-release/volumes/20240930/traces/'
}).result().read().result()

In [ ]:
i=2747 # ok the next one does NOT skip a bad gap, but it is 2748 -- is this allowed?
np.all(infer_source[i]['timeseries_input'][0,1]==x[i+1])

In [ ]:
2063 2420
((650, 2421),)
# good so this needs to be cut a bit to get it correct!

In [ ]:
all_idx = data_utils.adjust_valid_timesteps_for_split(valid_timesteps, 'test_holdout', 4)
print(all_idx[-36])

In [ ]:
def filter_infer_indices(x: list[int], context: int) -> dict[int, int]:
  assert context > 0, f'context needs to be > 0, but got {context}'
  valid_infer_idx, i_init = [], 0
  for i in range(1, len(x) + 1):
    if i == len(x) or x[i] != x[i - 1] + 1:
      i_len = i - i_init
      if i_len >= context:
        valid_infer_idx.extend(x[i_init:i - context + 1])
      i_init = i
  return valid_infer_idx

In [ ]:
print(filter_infer_indices(all_idx, 36))